In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [32]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python|json|regex",
        "solution_criteria": "Key criteria for evalutating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text.strip()) 

In [33]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [37]:
def grade_by_model(test_case, model):
    # Model Grader Function
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {model}
    Solution Critera: <criteria>{test_case["solution_criteria"]}<criteria>
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    
    return json.loads(eval_text.strip())


In [ ]:
# Functions to validate the output structure
import re
import ast

# Code Grader Functions
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [30]:
def run_prompt(test_case):
#  Merges the prompt and test case input, then returns the result
    prompt = f"""
        Please solve the following task:
   
        {test_case['task']}
        
        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code") # 'code' denotes wildcard placeholder for any format in Claude
    output = chat(messages, stop_sequences=["```"])
    return output

def run_test_case(test_case):
   # Calls run prompt, then grades the result
   
    output = run_prompt(test_case) # test case results
   
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade.get("score", 0)
    reasoning = model_grade.get("reasoning", "")
   
    syntax_score = grade_syntax(output, test_case)
   
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }    


def run_eval(dataset):
   # Loads the dataset and calls run_test_case with each case
   
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = sum(result["score"] for result in results) / len(results)
    print(f"Average Score: {average_score:.2f}")
    
    return results
        
        

In [39]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)
print(json.dumps(results, indent=2))


Average Score: 8.25
[
  {
    "output": "\nimport re\n\ndef parse_s3_bucket_name(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    if match:\n        return match.group(1)\n    return None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a given S3 URI (e.g., 's3://my-bucket-name/path/to/object')",
      "format": "regex",
      "solution_criteria": "The regex should correctly extract the bucket name from various S3 URI formats, handling different path depths and special characters in bucket names"
    },
    "score": 8.0,
    "reasoning": "The solution works correctly for standard, well-formed S3 URIs and demonstrates solid understanding of regex basics. However, it lacks robustness for production use. It doesn't validate S3 bucket naming conventions per AWS specifications, and omits defensive programming practices for invalid inputs. The regex itself is sound but the overall solution needs stricter validation and error handling."
  },
  {
    "output"